# 02 Association Rule Mining
Apriori and FP-Growth experiments with threshold grid search.

In [1]:
import warnings
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore")
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "utils" / "utils.py").exists():
    if (PROJECT_ROOT / "notebooks" / "utils" / "utils.py").exists():
        PROJECT_ROOT = PROJECT_ROOT / "notebooks"
    elif (PROJECT_ROOT.parent / "utils" / "utils.py").exists():
        PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from utils.utils import ensure_project_dirs, load_raw_dataset, clean_dataset, PROCESSED_DIR, REPORTS_DIR, FIGURES_DIR
from utils.features import engineer_kpis, build_post_feature_sets, aggregate_business_features
from utils.evaluation import regression_metrics, rank_models
from utils.visualization import set_plot_style, save_figure

set_plot_style()
ensure_project_dirs()
RAW_DATA_PATH = Path(r"C:\univ\Data mining\Project\synthetic_generator\synthetic_social_media_posts.csv")
if not RAW_DATA_PATH.exists():
    RAW_DATA_PATH = PROJECT_ROOT / "synthetic_generator" / "synthetic_social_media_posts.csv"
KPI_PATH = PROCESSED_DIR / "kpi_dataset.csv"

## Load KPI Dataset and Build Transaction Matrix

In [2]:
if KPI_PATH.exists():
    df = pd.read_csv(KPI_PATH, parse_dates=["post_date"])
else:
    df = engineer_kpis(clean_dataset(load_raw_dataset(RAW_DATA_PATH)))
df.head()

,business_name,sector,followers_count,post_date,posting_hour,day_of_week,month,post_type,caption_text,caption_length,...,is_reel,posting_time_bin,caption_length_bin,hashtags_count_bin,emoji_count_bin,engagement_level,business_size_bin,high_engagement,high_view_rate,high_comment_rate
0,Ramallah Care Pharmacy,pharmacy,13666,2025-12-16,22,Tuesday,12,carousel,صحتك بتهمنا Swipe وشوفوا الباقي. موجودين في Ra...,109,...,0,night,medium,few,none,low,large,0,0,0
1,Nablus Bites,restaurant,12838,2025-10-18,20,Saturday,10,video,اليوم عنا أكل طيب فيديو جديد. احنا جاهزين احجز...,78,...,0,evening,medium,few,none,medium,medium,0,0,0
2,Al Amal Dental,clinic,4652,2025-01-17,21,Friday,1,carousel,صحتكم أولويتنا سوايب وشوفوا التفاصيل. شو رأيكم...,142,...,0,night,long,few,few,low,small,0,0,0
3,Hebron Study Hub,education_center,26289,2025-03-27,20,Thursday,3,reel,استنونا بدورة جديدة شوفوا الريل. زورونا بفرعنا...,90,...,1,evening,medium,few,few,high,large,1,1,1
4,Bethlehem Brew Bar,cafe,2210,2025-11-01,22,Saturday,11,image,أجواء ولا أروع صورة جديدة. أهلا بالناس الحلوة ...,72,...,0,night,medium,few,none,medium,small,0,0,0


In [3]:
import importlib.util, subprocess
if importlib.util.find_spec("mlxtend") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "mlxtend", "-q"])
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules

tx = df[[
    "sector","post_type","posting_time_bin","caption_length_bin","hashtags_count_bin","emoji_count_bin","language",
    "CTA_present","promo_post","mentions_location","religious_theme","patriotic_theme","arabic_dialect_style",
]].copy()
tx["outcome_high_engagement"] = df["engagement_level"].eq("high")
tx["outcome_high_view_rate"] = df["high_view_rate"].eq(1)
tx["outcome_high_comment_rate"] = df["high_comment_rate"].eq(1)
for c in tx.columns:
    tx[c] = c + "=" + tx[c].astype(str)
X = pd.get_dummies(tx)

## Run Grid Experiments and Compare Rule Quality

In [4]:
supports = [0.02, 0.05, 0.1]
confs = [0.4, 0.6, 0.8]
lifts = [1.2, 1.5, 2.0]
algos = {"apriori": apriori, "fpgrowth": fpgrowth}
targets = {"outcome_high_engagement=True","outcome_high_view_rate=True","outcome_high_comment_rate=True"}
rows, all_rules = [], []

for algo_name, algo in algos.items():
    for s in supports:
        itemsets = algo(X, min_support=s, use_colnames=True, max_len=3)
        if itemsets.empty:
            continue
        for c in confs:
            try:
                rules = association_rules(itemsets, metric="confidence", min_threshold=c)
            except MemoryError:
                # Skip parameter combinations that explode in memory.
                continue
            except Exception:
                continue
            if rules.empty:
                continue
            rules = rules[rules["consequents"].apply(lambda r: any(t in r for t in targets))]
            if rules.empty:
                continue
            for l in lifts:
                r = rules[rules["lift"] >= l].copy()
                rows.append({
                    "algorithm": algo_name, "min_support": s, "min_confidence": c, "min_lift": l,
                    "rules_count": len(r), "avg_lift": r["lift"].mean() if len(r) else 0,
                    "avg_confidence": r["confidence"].mean() if len(r) else 0,
                    "avg_support": r["support"].mean() if len(r) else 0,
                })
                if len(r):
                    r["algorithm"], r["min_support"], r["min_confidence"], r["min_lift_threshold"] = algo_name, s, c, l
                    all_rules.append(r)

exp = pd.DataFrame(rows)
rule_export_cols = ["algorithm","min_support","min_confidence","min_lift_threshold","support","confidence","lift","antecedent_str","consequent_str"]
if exp.empty:
    ranked = pd.DataFrame(columns=["algorithm","min_support","min_confidence","min_lift","rules_count","avg_lift","avg_confidence","avg_support","composite_score"])
    best_rules = pd.DataFrame(columns=rule_export_cols + ["antecedents","consequents"])
    best_rules_by_sector = pd.DataFrame(columns=["sector","useful_rules","avg_lift","avg_confidence","top_rule"])
else:
    ranked = rank_models(exp, higher_is_better_cols=["rules_count","avg_lift","avg_confidence","avg_support"])
    best = ranked.iloc[0]
    rules_df = pd.concat(all_rules, ignore_index=True) if all_rules else pd.DataFrame()
    if rules_df.empty:
        best_rules = pd.DataFrame(columns=rule_export_cols + ["antecedents","consequents"])
    else:
        best_rules = rules_df[
            (rules_df["algorithm"] == best["algorithm"]) &
            (rules_df["min_support"] == best["min_support"]) &
            (rules_df["min_confidence"] == best["min_confidence"]) &
            (rules_df["lift"] >= best["min_lift"])
        ].copy()
        if not best_rules.empty:
            best_rules["antecedent_str"] = best_rules["antecedents"].apply(lambda s: ", ".join(sorted(s)))
            best_rules["consequent_str"] = best_rules["consequents"].apply(lambda s: ", ".join(sorted(s)))
        else:
            best_rules = pd.DataFrame(columns=rule_export_cols + ["antecedents","consequents"])

    sector_rows = []
    if not best_rules.empty and "antecedents" in best_rules.columns:
        for sector in sorted(df["sector"].unique()):
            marker = f"sector={sector}"
            sec = best_rules[best_rules["antecedents"].apply(lambda s: marker in s)]
            if len(sec) == 0:
                continue
            top = sec.sort_values("lift", ascending=False).iloc[0]
            sector_rows.append({
                "sector": sector,
                "useful_rules": len(sec),
                "avg_lift": sec["lift"].mean(),
                "avg_confidence": sec["confidence"].mean(),
                "top_rule": f"{top['antecedent_str']} -> {top['consequent_str']}",
            })
    best_rules_by_sector = pd.DataFrame(sector_rows)

best_rules.reindex(columns=rule_export_cols).to_csv(PROCESSED_DIR / "association_rules.csv", index=False)
best_rules_by_sector.to_csv(PROCESSED_DIR / "best_rules_by_sector.csv", index=False)
ranked.to_csv(REPORTS_DIR / "association_rule_experiments.csv", index=False)
display(ranked.head(12))
display(best_rules.reindex(columns=["support","confidence","lift","antecedent_str","consequent_str"]).head(20))


,algorithm,min_support,min_confidence,min_lift,rules_count,avg_lift,avg_confidence,avg_support,composite_score


,support,confidence,lift,antecedent_str,consequent_str
